# Multimodal RGB-Thermal Training on Google Colab (LLVIP)

Trains the **RGB-only** and **Thermal-only** YOLOv8 baselines on a Colab GPU, evaluates both on the test set, and packages the results for download.

**Before you start**
1. Upload the LLVIP dataset to your Google Drive (or use the optional zip-upload cell in Step 1). Expected structure:

```
LLVIP/
├── visible/train     # 12,025 RGB images
├── visible/test      # 3,463 RGB images
├── infrared/train    # 12,025 thermal images
├── infrared/test     # 3,463 thermal images
└── Annotations/      # 15,488 PASCAL VOC XML files
```

2. Set the runtime to GPU: **Runtime → Change runtime type → T4 GPU**.
3. Run the cells top to bottom.

Everything is written under `/content/multimodal-rgb-thermal/outputs/`, mirroring the local repo layout, so you can drag the downloaded folders straight back into the project.

**Time estimate:** on a T4, ~1–3 min/epoch at 640×640 → roughly 1.5–2.5 h per modality (3–5 h total). Free-tier GPU quota varies; if you run out, lower `EPOCHS` (e.g. 30) or `IMGSZ` (320, as used in the local CPU baselines) and re-run.


In [ ]:
!pip install -q ultralytics

import json
import random
import shutil
import time
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path

import torch
from ultralytics import YOLO
from google.colab import drive, files

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "No GPU detected. Switch to a GPU runtime: Runtime > Change runtime type > T4 GPU"


## Step 1 — Mount the dataset

Mount Google Drive and point `LLVIP_ROOT` at the LLVIP folder. The structure is verified before anything else runs.


In [ ]:
drive.mount("/content/drive")

LLVIP_ROOT = Path("/content/drive/MyDrive/LLVIP")

VISIBLE_TRAIN  = LLVIP_ROOT / "visible" / "train"
VISIBLE_TEST   = LLVIP_ROOT / "visible" / "test"
INFRARED_TRAIN = LLVIP_ROOT / "infrared" / "train"
INFRARED_TEST  = LLVIP_ROOT / "infrared" / "test"
ANNOTATIONS    = LLVIP_ROOT / "Annotations"

PROJECT_OUT = Path("/content/multimodal-rgb-thermal/outputs")

EPOCHS = 50
BATCH  = 16
IMGSZ  = 640
MODEL  = "yolov8n"
SEED   = 42

for d in [VISIBLE_TRAIN, VISIBLE_TEST, INFRARED_TRAIN, INFRARED_TEST, ANNOTATIONS]:
    assert d.exists(), f"Missing directory: {d}"

print("Dataset OK")
print(f"  RGB train/test:     {len(list(VISIBLE_TRAIN.iterdir()))} / {len(list(VISIBLE_TEST.iterdir()))}")
print(f"  Thermal train/test: {len(list(INFRARED_TRAIN.iterdir()))} / {len(list(INFRARED_TEST.iterdir()))}")
print(f"  Annotations:        {len(list(ANNOTATIONS.iterdir()))}")


**Optional:** if you did not put the dataset on Drive, upload the LLVIP zip here instead, then re-run the config cell above after changing `LLVIP_ROOT` to `/content/LLVIP`.


In [ ]:
uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")
LLVIP_ROOT = Path("/content/LLVIP")
print("Extracted to", LLVIP_ROOT)


## Step 2 — Convert PASCAL VOC XML to YOLO format

This reproduces the repo's `src/data/voc_to_yolo.py` exactly:
- train/val split with `random.seed(42)`, 10% val capped at 500 images — the **same split** as your local runs
- skips the 2 empty annotations (`100030`, `100033`) and the zero-width boxes (`020118`, `020236`) flagged by the audit
- converts the predefined test split alongside train/val


In [ ]:
def parse_voc(path):
    try:
        root = ET.parse(path).getroot()
        size = root.find("size")
        width = int(size.find("width").text) if size is not None and size.find("width") is not None else 1280
        height = int(size.find("height").text) if size is not None and size.find("height") is not None else 1024
        objects = []
        for obj in root.findall("object"):
            name_el = obj.find("name")
            bndbox = obj.find("bndbox")
            if name_el is None or bndbox is None:
                continue
            els = [bndbox.find(t) for t in ("xmin", "ymin", "xmax", "ymax")]
            if any(e is None for e in els):
                continue
            objects.append({"name": name_el.text, "bbox": [int(e.text) for e in els]})
        return {"width": width, "height": height, "objects": objects}
    except Exception:
        return None


def voc_to_yolo_bbox(bbox, img_w, img_h):
    xmin, ymin, xmax, ymax = bbox
    if xmin >= xmax or ymin >= ymax:
        return None
    w, h = xmax - xmin, ymax - ymin
    return [(xmin + w / 2) / img_w, (ymin + h / 2) / img_h, w / img_w, h / img_h]


def split_files(image_paths, val_ratio=0.1, max_val=500, seed=42):
    random.seed(seed)
    n = len(image_paths)
    val_count = min(max(int(n * val_ratio), 1 if n > 0 else 0), max_val)
    val_set = set(random.sample(range(n), val_count)) if n > 0 else set()
    train = [p for i, p in enumerate(image_paths) if i not in val_set]
    val = [p for i, p in enumerate(image_paths) if i in val_set]
    return train, val


def convert_dataset(image_dir, annotation_dir, output_dir, test_image_dir=None, val_ratio=0.1, max_val=500, seed=42):
    image_dir, annotation_dir, output_dir = Path(image_dir), Path(annotation_dir), Path(output_dir)
    for split in ("train", "val", "test"):
        (output_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (output_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    stats = {"total_images": 0, "converted": 0, "test_converted": 0, "skipped_empty": 0,
             "skipped_invalid_bbox": 0, "skipped_unmatched": 0, "errors": 0}

    def process(img_path, img_target, lbl_target, is_test=False):
        stem = img_path.stem
        stats["total_images"] += 1
        ann_path = annotation_dir / f"{stem}.xml"
        if not ann_path.exists():
            stats["skipped_unmatched"] += 1
            return
        ann = parse_voc(ann_path)
        if ann is None:
            stats["errors"] += 1
            return
        if not ann["objects"]:
            stats["skipped_empty"] += 1
            return
        lines, valid = [], False
        for obj in ann["objects"]:
            yolo_bbox = voc_to_yolo_bbox(obj["bbox"], ann["width"], ann["height"])
            if yolo_bbox is None:
                stats["skipped_invalid_bbox"] += 1
                continue
            lines.append("0 " + " ".join(f"{v:.6f}" for v in yolo_bbox))
            valid = True
        if not valid:
            stats["skipped_invalid_bbox"] += 1
            return
        shutil.copy2(img_path, img_target / img_path.name)
        (lbl_target / f"{stem}.txt").write_text("\n".join(lines) + "\n")
        if is_test:
            stats["test_converted"] += 1
        else:
            stats["converted"] += 1

    image_files = sorted(f for f in image_dir.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
    train_paths, val_paths = split_files(image_files, val_ratio, max_val, seed)
    for f in train_paths:
        process(f, output_dir / "images" / "train", output_dir / "labels" / "train")
    for f in val_paths:
        process(f, output_dir / "images" / "val", output_dir / "labels" / "val")
    if test_image_dir is not None:
        test_files = sorted(f for f in Path(test_image_dir).iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
        for f in test_files:
            process(f, output_dir / "images" / "test", output_dir / "labels" / "test", is_test=True)

    stats["train_images"] = len(list((output_dir / "images" / "train").iterdir()))
    stats["val_images"] = len(list((output_dir / "images" / "val").iterdir()))
    stats["test_images"] = len(list((output_dir / "images" / "test").iterdir()))
    return stats


def create_yaml(dataset_dir, num_classes=1, class_names=None):
    if class_names is None:
        class_names = ["person"]
    content = f"path: {dataset_dir}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnc: {num_classes}\nnames: {class_names}\n"
    yaml_path = Path(dataset_dir) / "data.yaml"
    yaml_path.write_text(content)
    return yaml_path


def verify_conversion(dataset_dir):
    dataset_dir = Path(dataset_dir)
    stats = {}
    for split in ("train", "val", "test"):
        img_dir = dataset_dir / "images" / split
        lbl_dir = dataset_dir / "labels" / split
        imgs = {f.stem for f in img_dir.iterdir()} if img_dir.exists() else set()
        lbls = {f.stem for f in lbl_dir.iterdir()} if lbl_dir.exists() else set()
        stats[f"{split}_images"] = len(imgs)
        stats[f"{split}_labels"] = len(lbls)
        stats[f"{split}_orphan_images"] = len(imgs - lbls)
        stats[f"{split}_orphan_labels"] = len(lbls - imgs)
    return stats


In [ ]:
RGB_DS = PROJECT_OUT / "rgb" / MODEL / "yolo_dataset"

stats = convert_dataset(
    image_dir=VISIBLE_TRAIN,
    test_image_dir=VISIBLE_TEST,
    annotation_dir=ANNOTATIONS,
    output_dir=RGB_DS,
)
create_yaml(RGB_DS)
print("RGB conversion stats:", json.dumps(stats, indent=2))
print("RGB verification:", verify_conversion(RGB_DS))


In [ ]:
THERMAL_DS = PROJECT_OUT / "thermal" / MODEL / "yolo_dataset"

stats = convert_dataset(
    image_dir=INFRARED_TRAIN,
    test_image_dir=INFRARED_TEST,
    annotation_dir=ANNOTATIONS,
    output_dir=THERMAL_DS,
)
create_yaml(THERMAL_DS)
print("Thermal conversion stats:", json.dumps(stats, indent=2))
print("Thermal verification:", verify_conversion(THERMAL_DS))


## Step 3 — Train both baselines

Settings match the repo defaults (`src/config.py`): YOLOv8n, COCO-pretrained, seed 42. Weights, per-epoch `results.csv`, and training plots land in `outputs/{rgb,thermal}/yolov8n/train/` exactly as the local scripts produce.

If you hit an out-of-memory error, lower `BATCH` (8) or `IMGSZ` (320).


In [ ]:
model = YOLO(f"{MODEL}.pt")
model.train(
    data=str(RGB_DS / "data.yaml"),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    project=str(PROJECT_OUT / "rgb" / MODEL),
    name="train",
    seed=SEED,
    device=0,
    workers=4,
    pretrained=True,
    save=True,
    plots=True,
    exist_ok=True,
)


In [ ]:
model = YOLO(f"{MODEL}.pt")
model.train(
    data=str(THERMAL_DS / "data.yaml"),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    project=str(PROJECT_OUT / "thermal" / MODEL),
    name="train",
    seed=SEED,
    device=0,
    workers=4,
    pretrained=True,
    save=True,
    plots=True,
    exist_ok=True,
)


## Step 4 — Evaluate on the test set

Replicates `src/evaluation/evaluate_yolo.py`: precision, recall, F1, mAP@50, mAP@50:95, plus average inference time over a 100-image sample. Writes `test_results.json` / `test_results.csv` into each experiment folder.

Note: inference time here is measured on the **GPU**, so it is not comparable to the CPU numbers in your README baselines — only the detection metrics are.


In [ ]:
def evaluate(modality, yaml_path, test_img_dir, out_base):
    weights = out_base / "train" / "weights" / "best.pt"
    if not weights.exists():
        weights = out_base / "train" / "weights" / "last.pt"
    assert weights.exists(), f"No weights found in {out_base / 'train' / 'weights'}"

    model = YOLO(str(weights))
    res = model.val(
        data=str(yaml_path),
        split="test",
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,
        workers=4,
        conf=0.25,
        iou=0.5,
        save_json=True,
        save=False,
    )
    d = res.results_dict
    precision = d.get("metrics/precision(B)")
    recall = d.get("metrics/recall(B)")
    f1 = 2 * precision * recall / (precision + recall) if precision and recall and (precision + recall) > 0 else None

    metrics = {
        "modality": modality,
        "model": MODEL,
        "timestamp": datetime.now().isoformat(),
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "mAP50": d.get("metrics/mAP50(B)"),
        "mAP50_95": d.get("metrics/mAP50-95(B)"),
    }

    sample = sorted(test_img_dir.iterdir())[:100]
    times = []
    for img in sample:
        t0 = time.time()
        model.predict(str(img), imgsz=IMGSZ, verbose=False, conf=0.25)
        times.append(time.time() - t0)
    metrics["inference_time"] = {
        "avg_per_image_ms": round(sum(times) / len(times) * 1000, 2),
        "samples": len(times),
    }

    (out_base / "test_results.json").write_text(json.dumps(metrics, indent=2))
    print(f"Test Set Metrics ({modality.upper()} + {MODEL}):")
    print(json.dumps(metrics, indent=2))
    return metrics


In [ ]:
rgb_metrics = evaluate("rgb", RGB_DS / "data.yaml", VISIBLE_TEST, PROJECT_OUT / "rgb" / MODEL)


In [ ]:
thermal_metrics = evaluate("thermal", THERMAL_DS / "data.yaml", INFRARED_TEST, PROJECT_OUT / "thermal" / MODEL)


## Step 5 — Download the results

Zips the entire `outputs/` tree — weights (`best.pt`, `last.pt`), per-epoch `results.csv`, training plots, and `test_results.json/csv` — in the same layout as the local repo.

**Back on your machine:** extract the zip into `multimodal-rgb-thermal/` so the contents land under `outputs/`, then run `pip install -r requirements.txt` (CPU torch is enough) and `python -m src.evaluation.compare` to get the RGB-vs-thermal comparison table.


In [ ]:
shutil.make_archive("/content/llvip_training_results", "zip", PROJECT_OUT)
files.download("/content/llvip_training_results.zip")
